In [8]:
# Imports

import os
from dotenv import load_dotenv
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# load_dotenv()

# TOKEN = os.getenv("AQICN_TOKEN")
CITY = "Lahore"
DAYS_TO_FETCH    = 90


In [9]:
# fetching air quality data from open meteo
def get_city_coordinates(city_name):
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={city_name}&count=1&format=json"
    response = requests.get(url)
    data = response.json()
    if "results" in data:
        return data["results"][0]["latitude"], data["results"][0]["longitude"], data["results"][0]["name"]
    else:
        raise ValueError(f"City '{city_name}' not found.")

lat, lon, city = get_city_coordinates(CITY)
print(f"Coordinates for {city}: Lat={lat}, Lon={lon}")

def fetch_aqi_data_openmeteo(lat, lon, days):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    # meteo's api endpoint
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "us_aqi,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust",
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "timezone": "auto"
    }
    
    print(f"Air Quality data from {start_date.date()} to {end_date.date()}...")
    response = requests.get(url, params=params)
    data = response.json()
    
    if "hourly" not in data:
        raise ValueError(f"Open-Meteo AQI API Error: {data}")
    
    df = pd.DataFrame({
        "timestamp": pd.to_datetime(data["hourly"]["time"]),
        "us_aqi": data["hourly"]["us_aqi"],
        "pm25": data["hourly"]["pm2_5"],
        "pm10": data["hourly"]["pm10"],
        "co": data["hourly"]["carbon_monoxide"],
        "no2": data["hourly"]["nitrogen_dioxide"],
        "so2": data["hourly"]["sulphur_dioxide"],
        "o3": data["hourly"]["ozone"],
        "dust": data["hourly"]["dust"]
    })
    
    return df

df_aqi = fetch_aqi_data_openmeteo(lat, lon, DAYS_TO_FETCH)
print(f"Fetched {len(df_aqi)} records.")
display(df_aqi.head(10))

Coordinates for Lahore: Lat=31.558, Lon=74.35071
Air Quality data from 2026-04-27 to 2026-07-26...
Fetched 2184 records.


,timestamp,us_aqi,pm25,pm10,co,no2,so2,o3,dust
0,2026-04-27 00:00:00,77,20.4,27.1,467.0,30.7,6.6,61.0,11.0
1,2026-04-27 01:00:00,76,22.8,30.3,422.0,31.7,6.8,56.0,12.0
2,2026-04-27 02:00:00,74,24.0,31.5,395.0,32.7,7.0,51.0,13.0
3,2026-04-27 03:00:00,73,25.1,32.1,387.0,34.1,7.2,41.0,12.0
4,2026-04-27 04:00:00,72,26.4,32.5,396.0,35.6,7.4,32.0,10.0
5,2026-04-27 05:00:00,71,29.7,38.9,446.0,35.1,7.4,43.0,15.0
6,2026-04-27 06:00:00,70,34.0,43.7,588.0,35.6,8.0,58.0,14.0
7,2026-04-27 07:00:00,69,38.9,45.5,769.0,36.0,9.0,80.0,13.0
8,2026-04-27 08:00:00,69,27.0,35.0,858.0,33.3,9.8,104.0,13.0
9,2026-04-27 09:00:00,68,18.3,27.1,768.0,24.8,10.6,131.0,15.0


In [10]:
# getting weather data for accurate prediction of next days as current air quality not enough to predict future.
def fetch_weather_data(lat, lon, days):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation",
        "timezone": "auto"
    }
    
    print(f"Weather data from {start_date.date()} to {end_date.date()}...")
    response = requests.get(url, params=params)
    data = response.json()
    
    df = pd.DataFrame({
        "timestamp": pd.to_datetime(data["hourly"]["time"]),
        "temperature_c": data["hourly"]["temperature_2m"],
        "humidity_pct": data["hourly"]["relative_humidity_2m"],
        "wind_speed_kmh": data["hourly"]["wind_speed_10m"],
        "precipitation_mm": data["hourly"]["precipitation"]
    })
    
    return df

df_weather = fetch_weather_data(lat, lon, DAYS_TO_FETCH)

# merge aqi and weather df for 1 single df
df_raw = pd.merge(df_aqi, df_weather, on="timestamp", how="inner")
df_raw = df_raw.sort_values("timestamp").reset_index(drop=True)

print(f"Final raw dataset shape: {df_raw.shape}")
display(df_raw.head(10))

Weather data from 2026-04-27 to 2026-07-26...
Final raw dataset shape: (2184, 13)


,timestamp,us_aqi,pm25,pm10,co,no2,so2,o3,dust,temperature_c,humidity_pct,wind_speed_kmh,precipitation_mm
0,2026-04-27 00:00:00,77,20.4,27.1,467.0,30.7,6.6,61.0,11.0,30.3,31,7.8,0.0
1,2026-04-27 01:00:00,76,22.8,30.3,422.0,31.7,6.8,56.0,12.0,29.2,34,7.3,0.0
2,2026-04-27 02:00:00,74,24.0,31.5,395.0,32.7,7.0,51.0,13.0,28.0,37,6.0,0.0
3,2026-04-27 03:00:00,73,25.1,32.1,387.0,34.1,7.2,41.0,12.0,27.3,38,4.4,0.0
4,2026-04-27 04:00:00,72,26.4,32.5,396.0,35.6,7.4,32.0,10.0,27.1,37,5.1,0.0
5,2026-04-27 05:00:00,71,29.7,38.9,446.0,35.1,7.4,43.0,15.0,27.2,38,1.8,0.0
6,2026-04-27 06:00:00,70,34.0,43.7,588.0,35.6,8.0,58.0,14.0,26.8,38,3.2,0.0
7,2026-04-27 07:00:00,69,38.9,45.5,769.0,36.0,9.0,80.0,13.0,28.1,35,5.9,0.0
8,2026-04-27 08:00:00,69,27.0,35.0,858.0,33.3,9.8,104.0,13.0,30.6,31,3.1,0.0
9,2026-04-27 09:00:00,68,18.3,27.1,768.0,24.8,10.6,131.0,15.0,33.6,25,2.2,0.0


In [11]:
# feature engineerinh
# get features from timestamp to get periodic patterns
df_raw["hour"] = df_raw["timestamp"].dt.hour
df_raw["day"] = df_raw["timestamp"].dt.day
df_raw["month"] = df_raw["timestamp"].dt.month
df_raw["dayofweek"] = df_raw["timestamp"].dt.dayofweek  # 0=Monday - 6=Sunday

# reading of aqi and pm25 for an hour and day ago repesented by lag (lagging)
df_raw["aqi_lag_1"] = df_raw["us_aqi"].shift(1)
df_raw["aqi_lag_24"] = df_raw["us_aqi"].shift(24)
df_raw["pm25_lag_1"] = df_raw["pm25"].shift(1)
df_raw["pm25_lag_1"] = df_raw["pm25"].shift(24)

# how much aqi changed from previous hour
df_raw["aqi_change_rate"] = df_raw["us_aqi"] - df_raw["aqi_lag_1"]

# rolling average 
df_raw["pm25_rolling_24h"] = df_raw["pm25"].rolling(window=24, min_periods=1).mean()

# drop rows with nan entries for clean dataset
df_features = df_raw.dropna().reset_index(drop=True)

print(f"Original rows: {len(df_raw)} | Final  rows: {len(df_features)}")
print("New Dataset Columns:")
print(df_features.columns.tolist())
display(df_features.head(10))

Original rows: 2184 | Final  rows: 2160
New Dataset Columns:
['timestamp', 'us_aqi', 'pm25', 'pm10', 'co', 'no2', 'so2', 'o3', 'dust', 'temperature_c', 'humidity_pct', 'wind_speed_kmh', 'precipitation_mm', 'hour', 'day', 'month', 'dayofweek', 'aqi_lag_1', 'aqi_lag_24', 'pm25_lag_1', 'aqi_change_rate', 'pm25_rolling_24h']


,timestamp,us_aqi,pm25,pm10,co,no2,so2,o3,dust,temperature_c,...,precipitation_mm,hour,day,month,dayofweek,aqi_lag_1,aqi_lag_24,pm25_lag_1,aqi_change_rate,pm25_rolling_24h
0,2026-04-28 00:00:00,82,42.0,62.6,880.0,55.0,14.1,47.0,40.0,32.6,...,0.0,0,28,4,1,80.0,77.0,20.4,2.0,26.687500
1,2026-04-28 01:00:00,83,44.0,66.6,671.0,50.8,13.9,47.0,44.0,32.4,...,0.0,1,28,4,1,82.0,76.0,22.8,1.0,27.570833
2,2026-04-28 02:00:00,85,45.8,69.8,518.0,45.8,13.3,48.0,47.0,32.3,...,0.0,2,28,4,1,83.0,74.0,24.0,2.0,28.479167
3,2026-04-28 03:00:00,87,43.8,67.3,477.0,38.8,12.0,47.0,46.0,31.4,...,0.0,3,28,4,1,85.0,73.0,25.1,2.0,29.258333
4,2026-04-28 04:00:00,88,40.0,62.4,491.0,30.9,10.4,46.0,44.0,30.4,...,0.0,4,28,4,1,87.0,72.0,26.4,1.0,29.825000
5,2026-04-28 05:00:00,89,17.8,35.0,537.0,7.1,7.8,86.0,32.0,28.3,...,0.0,5,28,4,1,88.0,71.0,29.7,1.0,29.329167
6,2026-04-28 06:00:00,88,18.5,37.5,649.0,9.9,8.6,84.0,37.0,28.0,...,0.0,6,28,4,1,89.0,70.0,34.0,-1.0,28.683333
7,2026-04-28 07:00:00,87,19.8,39.6,794.0,13.7,9.6,82.0,44.0,28.6,...,0.0,7,28,4,1,88.0,69.0,38.9,-1.0,27.887500
8,2026-04-28 08:00:00,86,21.7,48.5,847.0,16.1,10.5,82.0,51.0,30.0,...,0.0,8,28,4,1,87.0,69.0,27.0,-1.0,27.666667
9,2026-04-28 09:00:00,85,21.4,52.9,717.0,16.2,11.0,83.0,58.0,30.8,...,0.0,9,28,4,1,86.0,68.0,18.3,-1.0,27.795833
